# Results Presentation Notebook

This notebook loads the aggregated analysis outputs, displays the current model summary table, and creates presentation-ready seaborn figures.

Primary inputs:
- `outputs/summary_by_model.csv`
- `outputs/summary_by_context.csv`
- `outputs/summary_by_dimension.csv`
- `outputs/pairwise_differences.csv`
- `outputs/results_master.csv`


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / 'outputs' / 'results_master.csv').exists() and (BASE_DIR.parent / 'outputs' / 'results_master.csv').exists():
    BASE_DIR = BASE_DIR.parent
OUTPUTS_DIR = BASE_DIR / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_MASTER_PATH = OUTPUTS_DIR / 'results_master.csv'
SUMMARY_MODEL_PATH = OUTPUTS_DIR / 'summary_by_model.csv'
SUMMARY_CONTEXT_PATH = OUTPUTS_DIR / 'summary_by_context.csv'
SUMMARY_DIMENSION_PATH = OUTPUTS_DIR / 'summary_by_dimension.csv'
PAIRWISE_PATH = OUTPUTS_DIR / 'pairwise_differences.csv'

required_paths = [
    RESULTS_MASTER_PATH,
    SUMMARY_MODEL_PATH,
    SUMMARY_CONTEXT_PATH,
    SUMMARY_DIMENSION_PATH,
    PAIRWISE_PATH,
]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required analysis files: {missing}')

print(f'Python: {sys.version.split()[0]}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'seaborn: {sns.__version__}')
if sys.version_info >= (3, 14):
    print('Warning: Python 3.14 with matplotlib 3.9 may raise RecursionError in seaborn plots. If that happens, run this notebook in Python 3.10-3.12.')

sns.set_theme(style='whitegrid', context='talk')


In [ ]:
results_master = pd.read_csv(RESULTS_MASTER_PATH)
summary_by_model = pd.read_csv(SUMMARY_MODEL_PATH)
summary_by_context = pd.read_csv(SUMMARY_CONTEXT_PATH)
summary_by_dimension = pd.read_csv(SUMMARY_DIMENSION_PATH)
pairwise_differences = pd.read_csv(PAIRWISE_PATH)

model_summary = summary_by_model[summary_by_model['record_type'].fillna('') == 'asc'].copy()
context_summary = summary_by_context[summary_by_context['record_type'].fillna('') == 'asc'].copy()
dimension_summary = summary_by_dimension[summary_by_dimension['record_type'].fillna('') == 'asc'].copy()

current_model_summary = (
    model_summary[[
        'model',
        'mean_toxicity',
        'mean_helpfulness_score',
        'mean_factuality_score',
        'mean_stereotype_score',
    ]]
    .rename(columns={
        'mean_helpfulness_score': 'mean_helpfulness',
        'mean_factuality_score': 'mean_factuality',
        'mean_stereotype_score': 'mean_stereotype',
    })
    .sort_values('mean_toxicity')
    .reset_index(drop=True)
)

metric_cols = ['mean_toxicity', 'mean_helpfulness', 'mean_factuality', 'mean_stereotype']
current_model_summary[metric_cols] = current_model_summary[metric_cols].round(4)

print('Current model summary:')
display(current_model_summary)


In [ ]:
def save_current_figure(filename: str):
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Saved: {path}')


def annotate_bars(ax, fmt: str = '{:.3f}'):
    for patch in ax.patches:
        height = patch.get_height()
        if pd.isna(height):
            continue
        x = patch.get_x() + patch.get_width() / 2
        offset = max(height * 0.02, 0.002) if height >= 0 else -max(abs(height) * 0.08, 0.002)
        va = 'bottom' if height >= 0 else 'top'
        ax.text(x, height + offset, fmt.format(height), ha='center', va=va, fontsize=10)


## 1. Average Toxicity Score by Model


In [ ]:
tox_plot = current_model_summary.sort_values('mean_toxicity', ascending=True)
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=tox_plot, x='model', y='mean_toxicity', hue='model', palette='rocket', legend=False)
ax.set_title('Average Toxicity Score by Model')
ax.set_xlabel('Model')
ax.set_ylabel('Mean toxicity')
annotate_bars(ax, fmt='{:.4f}')
save_current_figure('figure_1_average_toxicity_score.png')
plt.show()
tox_plot


## 2. Model Profile Heatmap


In [ ]:
heatmap_df = current_model_summary.set_index('model')[['mean_toxicity', 'mean_helpfulness', 'mean_factuality', 'mean_stereotype']]
plt.figure(figsize=(8, 4.8))
ax = sns.heatmap(heatmap_df, annot=True, fmt='.4f', cmap='YlGnBu', linewidths=0.5, cbar_kws={'label': 'Score'})
ax.set_title('Current Model Summary Heatmap')
ax.set_xlabel('Metric')
ax.set_ylabel('Model')
save_current_figure('figure_2_model_profile_heatmap.png')
plt.show()
heatmap_df


## 3. Helpfulness, Factuality, and Stereotype Comparison


In [ ]:
profile_long = current_model_summary.melt(
    id_vars='model',
    value_vars=['mean_helpfulness', 'mean_factuality', 'mean_stereotype'],
    var_name='metric',
    value_name='score',
)

metric_order = ['mean_helpfulness', 'mean_factuality', 'mean_stereotype']
plt.figure(figsize=(10, 5.5))
ax = sns.barplot(data=profile_long, x='metric', y='score', hue='model', order=metric_order, palette='Set2')
ax.set_title('Comparison of Core Quality Metrics by Model')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_xticklabels(['Helpfulness', 'Factuality', 'Stereotype'])
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
save_current_figure('figure_3_core_quality_metrics.png')
plt.show()
profile_long


## 4. Fairness Gap Summary by Model


In [ ]:
gap_summary = model_summary[[
    'model',
    'avg_toxicity_diff_autistic_minus_neurotypical',
    'avg_helpfulness_score_diff_autistic_minus_neurotypical',
    'avg_stereotype_score_diff_autistic_minus_neurotypical',
]].rename(columns={
    'avg_toxicity_diff_autistic_minus_neurotypical': 'toxicity_gap',
    'avg_helpfulness_score_diff_autistic_minus_neurotypical': 'helpfulness_gap',
    'avg_stereotype_score_diff_autistic_minus_neurotypical': 'stereotype_gap',
})

gap_long = gap_summary.melt(id_vars='model', var_name='metric', value_name='gap')
plt.figure(figsize=(11, 5.5))
ax = sns.barplot(data=gap_long, x='metric', y='gap', hue='model', palette='coolwarm')
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Average Fairness Gap by Model (Autistic Minus Neurotypical)')
ax.set_xlabel('Gap metric')
ax.set_ylabel('Average gap')
ax.set_xticklabels(['Toxicity gap', 'Helpfulness gap', 'Stereotype gap'])
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
save_current_figure('figure_4_fairness_gap_summary.png')
plt.show()
gap_summary.round(4)


## 5. Context-Level Toxicity Heatmap


In [ ]:
context_tox = context_summary[['context', 'model', 'mean_toxicity']].dropna().copy()
context_pivot = context_tox.pivot_table(index='context', columns='model', values='mean_toxicity', aggfunc='mean')
plt.figure(figsize=(9, max(5, 0.55 * len(context_pivot))))
ax = sns.heatmap(context_pivot, annot=True, fmt='.4f', cmap='mako', linewidths=0.4, cbar_kws={'label': 'Mean toxicity'})
ax.set_title('Context-Level Average Toxicity by Model')
ax.set_xlabel('Model')
ax.set_ylabel('Context')
save_current_figure('figure_5_context_toxicity_heatmap.png')
plt.show()
context_pivot


## 6. Dimension-Level Toxicity Ranking


In [ ]:
dimension_tox = (
    dimension_summary.groupby('dimension', as_index=False)['mean_toxicity']
    .mean()
    .sort_values('mean_toxicity', ascending=False)
)

plt.figure(figsize=(10, 5.5))
ax = sns.barplot(data=dimension_tox, x='mean_toxicity', y='dimension', hue='dimension', palette='viridis', legend=False)
ax.set_title('Average Toxicity by Evaluation Dimension')
ax.set_xlabel('Mean toxicity')
ax.set_ylabel('Dimension')
save_current_figure('figure_6_dimension_toxicity_ranking.png')
plt.show()
dimension_tox.round(4)
